# Baseline 추론 노트북

이 노트북은 입력 데이터셋에 대해 고정된 Vision-Language Model을 실행하고, 제출용 예측 파일을 다음 형식으로 저장합니다.

```text
sample_id,label
```

## 1. 환경 설정

아래 예시처럼 가상환경을 만들고 baseline 의존성을 설치합니다.

`baseline_requirements.txt`는 [baseline_requirements.txt 다운로드 링크](https://cfiles.dacon.co.kr/competitions/236722/baseline_requirements.txt)에서 받을 수 있습니다.

```bash
conda create -n challenge_env python=3.10 -y
conda activate challenge_env
pip install -r baseline_requirements.txt
```

Jupyter Notebook에서 이 가상환경을 선택하려면 커널 등록도 필요합니다.

```bash
pip install ipykernel
python -m ipykernel install --user --name challenge_env --display-name "Python (challenge_env)"
```

위 명령은 현재 활성화된 `challenge_env` 환경을 Jupyter의 커널 목록에 등록합니다.  
등록 후 노트북에서 `Kernel -> Change Kernel -> Python (challenge_env)`를 선택하면 이 환경으로 실행됩니다.

## 2. 라이브러리 불러오기

In [1]:
import json
import os
import time
from tqdm.auto import tqdm
from dataclasses import asdict
from pathlib import Path
from typing import Literal, NamedTuple

import torch
from vllm import LLM, EngineArgs, SamplingParams
from vllm.utils import FlexibleArgumentParser

import base64
import pandas as pd
from PIL import Image
from io import BytesIO

from pydantic import BaseModel
from vllm.sampling_params import GuidedDecodingParams

# GPU 사용 모드 선택: "all" 또는 "gpu0"
#GPU_MODE = "all"   # 전체 GPU 사용
GPU_MODE = "gpu0"  # 0번 GPU만 사용

if GPU_MODE == "gpu0":
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
elif GPU_MODE == "all":
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)
else:
    raise ValueError("GPU_MODE must be 'all' or 'gpu0'")

/home/dacon/anaconda3/envs/challenge_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 05-27 16:09:58 [__init__.py:239] Automatically detected platform cuda.


2026-05-27 16:09:59,764	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


## 3. 출력 형식 정의

In [2]:
class ReasonAnswer(BaseModel):
    reason: str 
    answer_id: Literal["0", "1", "2"]


class ModelRequestData(NamedTuple):
    engine_args: EngineArgs
    prompts: list[str]
    stop_token_ids: list[int] | None = None

## 4. 모델 요청 구성

In [3]:
def run_llava_onevision(questions: list[str]) -> ModelRequestData:

    prompts = [
            f"<|im_start|>user <image>\n{question}<|im_end|> \
        <|im_start|>assistant\n" for question in questions
        ]

    engine_args = EngineArgs(
        model="llava-hf/llava-onevision-qwen2-0.5b-si-hf", # 사용할 모델 경로 또는 Hugging Face 모델 ID
        max_model_len=16384, # 모델이 처리할 수 있는 최대 입력/출력 토큰 길이, 값이 클수록 긴 문맥 처리가 가능하지만, KV cache로 인해 VRAM 사용량이 증가
        limit_mm_per_prompt={"image": 1}, # 하나의 프롬프트에서 허용할 멀티모달 입력 개수 제한, image=1은 프롬프트당 이미지 1장만 허용
        tensor_parallel_size=torch.cuda.device_count(), # 모델을 분산 실행할 GPU 개수, 현재 사용 가능한 CUDA GPU 수만큼 tensor parallel 적용
        gpu_memory_utilization=0.9, # vLLM이 사용할 GPU 메모리 비율, 값이 높을수록 KV cache를 크게 잡지만, OOM 위험 증가
        disable_mm_preprocessor_cache=True, # 멀티모달 전처리 결과 캐시 비활성화, 반복 입력 캐시는 줄지만, 메모리 사용을 보수적으로 관리할 때 유용
    )

    return ModelRequestData(
        engine_args=engine_args,
        prompts=prompts,
    )

## 5. 유틸리티 함수

In [4]:
def parse_answers_field(raw: str):
    return json.loads(raw)

def normalize_answer_id(value):
    """0, 1, 2 중 하나의 라벨만 허용합니다."""
    if value is None:
        return "0"
    text = str(value).strip()
    return text if text in {"0", "1", "2"} else "0"

def load_image(image, img_size=512, base_64=False):
    try:
        if isinstance(image, (str, Path)):
            img = Image.open(str(image))
        else:
            img = Image.open(BytesIO(image["bytes"]))
        img = img.convert("RGB")
        width_percent = img_size / float(img.size[0])
        new_height = int((float(img.size[1]) * width_percent))

        img_resized = img.resize((img_size, new_height), Image.LANCZOS)

        if base_64:
            buffered = BytesIO()
            img_resized.save(buffered, format="JPEG")

            return base64.b64encode(buffered.getvalue()).decode('utf-8')
        else:
            return img_resized
    except Exception as e:
        print(e)
        return None  

## 6. 실행 인자 정의

In [5]:
def parse_args():
    parser = FlexibleArgumentParser(
        description='Demo on using vLLM for offline inference with '
        'vision language models for text generation')
    parser.add_argument('--model-type',
                        '-m',
                        type=str,
                        default="llava-onevision")
    parser.add_argument("--seed",
                        type=int,
                        default=42,
                        help="Set the seed when initializing `vllm.LLM`.")
    parser.add_argument(
        "--data-csv",
        type=str,
        default="/path/to/data.csv",
        help="Input csv with columns: sample_id,image_path,context,question,answers",
    )
    parser.add_argument(
        "--images-dir",
        type=str,
        default="/path/to/images",
        help="Directory storing images referenced by image_path",
    )
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--img-size", type=int, default=224)
    parser.add_argument("--max-samples", type=int, default=None)
    parser.add_argument("--output-path", type=str, default="./outputs/",
                        help="File path to store inference outputs CSV.")

    return parser.parse_args()

## 7. 메인 추론 함수

In [6]:
def main(args):
    modality = 'image'

    json_schema = ReasonAnswer.model_json_schema()
    guided_decoding_params_json = GuidedDecodingParams(json=json_schema)

    output_path = os.path.join(args.output_path)
    os.makedirs(output_path, exist_ok=True)

    df = pd.read_csv(args.data_csv)

    if "label" in df.columns:
        print("INFO: input csv has 'label' column, but inference never reads it for prediction.")

    df['model_output'] = None
    df['label'] = None

    if args.max_samples is not None:
        df = df.head(args.max_samples).copy()

    inputs = []
    batch_indices = []
    llm = None

    with tqdm(total=len(df), desc="Inference", unit="sample") as pbar:
        for row_idx, row in df.iterrows():
            image_path = Path(args.images_dir) / str(row["image_path"])

            context = 'Context: ' + str(row.get('context', ''))
            question = 'Question: ' + str(row.get('question', ''))

            answers = parse_answers_field(row["answers"])
            options = (
                'Options:\n'
                f'0. {answers[0]}\n'
                f'1. {answers[1]}\n'
                f'2. {answers[2]}\n'
            )

            pre_prompt = (
                "You are an expert Vision Language assistant. "
                "When given an image, a context, a question, and options, "
                "you MUST respond only with a JSON object"
            )

            post_prompt = (
                "Give the output in strict JSON format: "
                "{\n"
                "   \"reason\": \"One short sentence of reasoning.\",\n"
                "   \"answer_id\": \"<one of: 0, 1, 2>\"\n"
                "}\n"
            )

            rule_prompt = "Do NOT output multiple options."

            prompt_text = (
                pre_prompt + '\n' +
                context + '\n' +
                question + '\n' +
                options + '\n' +
                post_prompt + '\n' +
                rule_prompt
            )

            data = load_image(image_path, img_size=args.img_size)

            if data is None:
                df.at[row_idx, "label"] = "0"
                df.at[row_idx, "model_output"] = ""

                save_cols = ['sample_id', 'label']
                existing_cols = [c for c in save_cols if c in df.columns]
                df[existing_cols].to_csv(
                    os.path.join(output_path, 'baseline_submission.csv'),
                    index=False
                )

                pbar.update(1)
                continue

            req_data = run_llava_onevision([prompt_text])

            if llm is None:
                default_limits = {"image": 1, "video": 0, "audio": 0}
                req_data.engine_args.limit_mm_per_prompt = default_limits

                engine_args = asdict(req_data.engine_args) | {
                    "seed": args.seed,
                }

                llm = LLM(**engine_args)

            prompts = req_data.prompts[0]

            sampling_params = SamplingParams(
                temperature=0.0,
                max_tokens=128,
                stop_token_ids=req_data.stop_token_ids,
                guided_decoding=guided_decoding_params_json
            )

            inputs.append({
                "prompt": prompts,
                "multi_modal_data": {
                    modality: data
                }
            })

            batch_indices.append(row_idx)

            is_batch_ready = ((row_idx + 1) % args.batch_size) == 0
            is_last_row = row_idx == df.index[-1]

            if is_batch_ready or is_last_row:
                outputs = llm.generate(
                    inputs,
                    sampling_params=sampling_params,
                    use_tqdm=False,
                )

                for idx, o in zip(batch_indices, outputs):
                    generated_text = o.outputs[0].text
                    df.at[idx, 'model_output'] = generated_text

                    try:
                        json_match_start = generated_text.find("{")
                        json_match_end = generated_text.rfind("}")

                        if json_match_start >= 0 and json_match_end > json_match_start:
                            parsed = json.loads(
                                generated_text[json_match_start:json_match_end + 1]
                            )
                        else:
                            parsed = json.loads(generated_text)

                        df.at[idx, 'label'] = normalize_answer_id(parsed.get("answer_id"))

                    except Exception:
                        df.at[idx, 'label'] = "0"

                save_cols = ['sample_id', 'label']
                existing_cols = [c for c in save_cols if c in df.columns]
                df[existing_cols].to_csv(
                    os.path.join(output_path, 'baseline_submission.csv'),
                    index=False
                )

                pbar.update(len(batch_indices))

                inputs = []
                batch_indices = []

## 8. GPU 확인

In [7]:
def check_pytorch_gpu():
    try:
        if torch.cuda.is_available():
            print(f"PyTorch can access {torch.cuda.device_count()} GPU(s).")
            for i in range(torch.cuda.device_count()):
                print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        else:
            print("PyTorch cannot access any GPUs.")
    except Exception as e:
        print(f"An error occurred: {e}")

check_pytorch_gpu()

PyTorch can access 1 GPU(s).
GPU 0: NVIDIA GeForce RTX 3090


## 9. 노트북 실행 설정

아래 경로를 실행 환경에 맞게 수정한 뒤 추론을 실행합니다.

CSV의 `image_path`가 `./images/test_img_0000.jpg` 형식이면 `images_dir`에는 split 루트 폴더를 넣습니다. 예: `../processed_data/test`

In [8]:
from argparse import Namespace

args = Namespace(
    model_type="llava-onevision",
    seed=42,
    data_csv="./test/test.csv",
    images_dir="./test",
    batch_size=16, # Your batch size
    img_size=224,
    max_samples=None,
    output_path="./outputs/",
)

args

Namespace(model_type='llava-onevision', seed=42, data_csv='./test/test.csv', images_dir='./test', batch_size=16, img_size=224, max_samples=None, output_path='./outputs/')

## 10. Baseline 추론 실행

In [9]:
_t0 = time.time()
main(args)
print(f"total_elapsed_seconds={time.time() - _t0:.2f}")

Inference:   0%|                                                                                                                                                                                                                         | 0/8500 [00:00<?, ?sample/s]

INFO 05-27 16:10:09 [config.py:600] This model supports multiple tasks: {'reward', 'score', 'generate', 'classify', 'embed'}. Defaulting to 'generate'.
INFO 05-27 16:10:09 [config.py:1780] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 05-27 16:10:11 [utils.py:2273] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/getting_started/troubleshooting.html#python-multiprocessing for more information. Reason: CUDA is initialized
INFO 05-27 16:10:15 [__init__.py:239] Automatically detected platform cuda.
INFO 05-27 16:10:17 [core.py:61] Initializing a V1 LLM engine (v0.8.3) with config: model='llava-hf/llava-onevision-qwen2-0.5b-si-hf', speculative_config=None, tokenizer='llava-hf/llava-onevision-qwen2-0.5b-si-hf', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_se

2026-05-27 16:10:17,898 - INFO - flashinfer.jit: Prebuilt kernels not found, using JIT backend


WARNING 05-27 16:10:18 [utils.py:2413] Methods determine_num_available_blocks,device_config,get_cache_block_size_bytes,initialize_cache not implemented in <vllm.v1.worker.gpu_worker.Worker object at 0x7efeab1b7610>
INFO 05-27 16:10:18 [parallel_state.py:957] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, TP rank 0
INFO 05-27 16:10:18 [cuda.py:221] Using Flash Attention backend on V1 engine.
INFO 05-27 16:10:19 [gpu_model_runner.py:1258] Starting to load model llava-hf/llava-onevision-qwen2-0.5b-si-hf...
INFO 05-27 16:10:19 [config.py:3334] cudagraph sizes specified by model runner [1, 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192, 200, 208, 216, 224, 232, 240, 248, 256, 264, 272, 280, 288, 296, 304, 312, 320, 328, 336, 344, 352, 360, 368, 376, 384, 392, 400, 408, 416, 424, 432, 440, 448, 456, 464, 472, 480, 488, 496, 504, 512] is overridden by config [512, 384, 256, 128, 4, 2, 1, 392, 264, 136, 8, 400, 272, 

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  2.55it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  2.55it/s]



INFO 05-27 16:10:21 [loader.py:447] Loading weights took 0.48 seconds
INFO 05-27 16:10:21 [gpu_model_runner.py:1273] Model loading took 1.6822 GiB and 1.510393 seconds
INFO 05-27 16:10:21 [gpu_model_runner.py:1542] Encoder cache will be initialized with a budget of 8748 tokens, and profiled with 1 image items of the maximum feature size.


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


INFO 05-27 16:10:30 [backends.py:416] Using cache directory: /home/dacon/.cache/vllm/torch_compile_cache/d2e8637ba6/rank_0_0 for vLLM's torch.compile
INFO 05-27 16:10:30 [backends.py:426] Dynamo bytecode transform time: 5.38 s
INFO 05-27 16:10:31 [backends.py:115] Directly load the compiled graph for shape None from the cache
INFO 05-27 16:10:35 [monitor.py:33] torch.compile takes 5.38 s in total
INFO 05-27 16:10:35 [kv_cache_utils.py:578] GPU KV cache size: 1,195,184 tokens
INFO 05-27 16:10:35 [kv_cache_utils.py:581] Maximum concurrency for 16,384 tokens per request: 72.95x
INFO 05-27 16:10:59 [gpu_model_runner.py:1608] Graph capturing finished in 24 secs, took 1.23 GiB
INFO 05-27 16:10:59 [core.py:162] init engine (profile, create kv cache, warmup model) took 38.05 seconds


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/home/dacon/anaconda3/envs/challenge_env/lib/python3.10/site-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Inference: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8500/8500 [13:09<00:00, 10.76sample/s]
[rank0]:[W527 16:23:11.032175388 ProcessGroupNCCL.cpp:1496] Warning: WARNING: destroy_process_group() was not calle

total_elapsed_seconds=792.52


## 11. 메모리 정리

In [11]:
import gc
gc.collect()

0

## 12. 출력 파일

추론이 끝나면 `args.output_path` 아래에 결과 파일이 저장되며, 
추론 과정에서 모델 출력 결과로부터 파싱 실패로 예측값이 결측된 경우, 해당 예측은 일괄적으로 0으로 처리합니다.

- `baseline_submission.csv`
  - `sample_id,label`